In [1]:
from seismic import SeismicIndex

In [4]:
json_input_file = "/data1/knn_datasets/sparse_datasets/beir/beir_spladev3_fromCarlos_CROSSCHECK-MRR-BEFORE-USE/beir/arguana/docs/docs_anserini.jsonl"

In [5]:
index = SeismicIndex.build(
    json_input_file,
    n_postings=3000,
    centroid_fraction=0.1,
    min_cluster_size=2,
    summary_energy=0.4, 
    max_fraction=4.0,
    load_content=True)


Building the index...
Configuration { pruning: GlobalThreshold { n_postings: 3000, max_fraction: 4.0 }, blocking: RandomKmeans { centroid_fraction: 0.1, min_cluster_size: 2, clustering_algorithm: RandomKmeansInvertedIndexApprox { doc_cut: 15 } }, summarization: EnergyPreserving { summary_energy: 0.4 }, knn: KnnConfiguration { nknn: 0, knn_path: None } }
Reading the collection..
Number of rows: 8674
Elapsed time to read the collection: 1 secs
Distributing and pruning postings: 0 secs
Number of posting lists: 23496
Avg posting list length: 109.22
Building summaries: 1 secs


In [6]:
import numpy as np
import json

file_path = "/data1/knn_datasets/sparse_datasets/beir/beir_spladev3_fromCarlos_CROSSCHECK-MRR-BEFORE-USE/beir/arguana/queries/docs_anserini.jsonl"

queries = []
with open(file_path, 'r') as f:
    for line in f:
        queries.append(json.loads(line))

MAX_TOKEN_LEN = 30
string_type  = f'U{MAX_TOKEN_LEN}'

queries_ids = np.array([q['id'] for q in queries], dtype=string_type)

query_components = []
query_values = []

for query in queries:
    vector = query['vector']
    query_components.append(np.array(list(vector.keys()), dtype=string_type))
    query_values.append(np.array(list(vector.values()), dtype=np.float32))

In [7]:
results = index.batch_search(
    queries_ids=queries_ids,
    query_components=query_components,
    query_values=query_values,
    k=10,
    query_cut=10,
    heap_factor=0.7,
    n_knn=0,
    sorted=True, #specified even if default value
    num_threads=1,
)

In [10]:
# --- RAG: retrieve document texts for the top results ---
#
# Since the index was built with load_content=True, we can call
# index.get_doc_text(doc_id) to fetch the stored passage text.
#
# Example: build a context prompt for the first query

first_query_text = queries[0].get("text", queries[0].get("id", ""))
first_results = results[0]  # list of (query_id, score, doc_id) for query 0

context_passages = []
for query_id, score, doc_id in first_results:
    text = index.get_doc_text(doc_id)
    if text is not None:
        context_passages.append(f"[{doc_id}] {text}")

prompt = (
    f"Query: {first_query_text}\n\n"
    "Relevant passages:\n"
    + "\n\n".join(f"{i+1}. {p}" for i, p in enumerate(context_passages))
    + "\n\nAnswer based on the passages above:"
)

print(prompt)

Query: test-environment-aeghhgwpe-pro02a

Relevant passages:
1. [test-environment-aeghhgwpe-pro02a] animals environment general health health general weight philosophy ethics Being vegetarian helps the environment  Becoming a vegetarian is an environmentally friendly thing to do. Modern farming is one of the main sources of pollution in our rivers. Beef farming is one of the main causes of deforestation, and as long as people continue to buy fast food in their billions, there will be a financial incentive to continue cutting down trees to make room for cattle. Because of our desire to eat fish, our rivers and seas are being emptied of fish and many species are facing extinction. Energy resources are used up much more greedily by meat farming than my farming cereals, pulses etc. Eating meat and fish not only causes cruelty to animals, it causes serious harm to the environment and to biodiversity. For example consider Meat production related pollution and deforestation  At Toronto’s 1992